# 05 - Evaluate

Reports detection metrics on the **LettuceMOTS val** split and on **your own frames** as **two separate tables** - never a single merged accuracy across public and own data.

Needs the torch + ultralytics stack and a trained checkpoint.

In [ ]:
import os, sys
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "croprow" / "utils.py").is_file())
sys.path.insert(0, str(REPO_ROOT))
from croprow import utils as U
CW = REPO_ROOT / "croprow"
DATA_DIR = CW / "data"
MODELS_DIR = CW / "models"
RUNS_DIR = CW / "runs"
RESULTS_MD = CW / "RESULTS.md"

# ===================== CONFIG (edit here only) =====================
WEIGHTS       = str(MODELS_DIR / "best.pt")            # or best_finetuned.pt
LETTUCE_YAML  = str(DATA_DIR / "lettuce.yaml")         # from 01_dataset_prep
OWN_DATA_YAML = os.environ.get("OWN_DATA_YAML", "")   # "" = skip own-frames eval
IMGSZ         = 640
DEVICE        = 0
# ===================================================================
print("weights:", WEIGHTS)

## Environment check

In [ ]:
# This notebook needs the training/inference stack (torch + ultralytics),
# NOT installed in the light 01/02 env. Install into a Python 3.11 venv with
# numpy<2 -- see croprow/requirements-train.txt and croprow/README.md.
try:
    import torch
    from ultralytics import YOLO
    import ultralytics
    print("torch      :", torch.__version__)
    print("ultralytics:", ultralytics.__version__)
    print("CUDA avail :", torch.cuda.is_available(),
          "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"))
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        f"Missing training dependency: {e.name}. Install croprow/requirements-train.txt "
        "into a Python 3.11 (numpy<2) venv before running this notebook."
    ) from e

In [ ]:
if not Path(WEIGHTS).is_file():
    raise FileNotFoundError(f"Weights not found: {WEIGHTS}. Train first (03/04).")

def report(tag, data_yaml):
    m = YOLO(WEIGHTS).val(data=data_yaml, imgsz=IMGSZ, device=DEVICE, split="val")
    row = dict(mAP50=float(m.box.map50), map5095=float(m.box.map),
               precision=float(m.box.mp), recall=float(m.box.mr))
    print(f"\n=== {tag} ===")
    print(f"  mAP50    : {row['mAP50']:.4f}")
    print(f"  mAP50-95 : {row['map5095']:.4f}")
    print(f"  precision: {row['precision']:.4f}")
    print(f"  recall   : {row['recall']:.4f}")
    return row

## A) LettuceMOTS val (public data)

In [ ]:
lettuce = report("LettuceMOTS-val", LETTUCE_YAML)
U.append_results_row(RESULTS_MD, run=Path(WEIGHTS).stem + "-eval",
                     dataset="LettuceMOTS-val", map50=lettuce["mAP50"],
                     map5095=lettuce["map5095"], precision=lettuce["precision"],
                     recall=lettuce["recall"], epochs="-", imgsz=IMGSZ)

## B) Your own frames (reported SEPARATELY)

Runs only if `OWN_DATA_YAML` is set. Logged as its own row - the public and own-data numbers are never combined.

In [ ]:
if OWN_DATA_YAML and Path(OWN_DATA_YAML).is_file():
    own = report("own-frames", OWN_DATA_YAML)
    U.append_results_row(RESULTS_MD, run=Path(WEIGHTS).stem + "-eval",
                         dataset="own-frames", map50=own["mAP50"],
                         map5095=own["map5095"], precision=own["precision"],
                         recall=own["recall"], epochs="-", imgsz=IMGSZ)
else:
    print("OWN_DATA_YAML not set / not found -> skipping own-frames eval "
          "(this is expected until you supply your frames).")

## RESULTS.md

In [ ]:
print(RESULTS_MD.read_text())